In [1]:
# Cell 1 — Evaluation Pipeline

print("=" * 70)
print("AI RESUME & JOB MATCHING ASSISTANT")
print("EVALUATION PIPELINE")
print("=" * 70)

AI RESUME & JOB MATCHING ASSISTANT
EVALUATION PIPELINE


In [5]:
# Cell 2 — Import evaluation libraries

import json
import ast
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Evaluation libraries imported successfully.")

Evaluation libraries imported successfully.


In [6]:
# Cell 3 — Set project paths

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project paths configured successfully.")

print("\nProcessed data folder:")
print(PROCESSED_DIR.resolve())

print("\nResults folder:")
print(RESULTS_DIR.resolve())

Project paths configured successfully.

Processed data folder:
C:\Users\Me\OneDrive\Desktop\Resume project ( Gen AI-RAG)\data\processed

Results folder:
C:\Users\Me\OneDrive\Desktop\Resume project ( Gen AI-RAG)\results


In [7]:
# Cell 4 — Verify required evaluation files

required_files = [
    PROCESSED_DIR / "evaluation_pairs.csv",
    RESULTS_DIR / "tfidf_test_results.csv",
    RESULTS_DIR / "tfidf_validation_results.csv",
    RESULTS_DIR / "semantic_results.csv",
    RESULTS_DIR / "hybrid_matching_results.csv",
    RESULTS_DIR / "validated_rag_output.json"
]

print("Checking required files:\n")

for file_path in required_files:
    status = "✓ FOUND" if file_path.exists() else "✗ NOT FOUND"
    print(f"{status}: {file_path}")

Checking required files:

✓ FOUND: ..\data\processed\evaluation_pairs.csv
✓ FOUND: ..\results\tfidf_test_results.csv
✓ FOUND: ..\results\tfidf_validation_results.csv
✓ FOUND: ..\results\semantic_results.csv
✓ FOUND: ..\results\hybrid_matching_results.csv
✓ FOUND: ..\results\validated_rag_output.json


In [8]:
# Cell 5 — Load evaluation pairs

evaluation_df = pd.read_csv(
    PROCESSED_DIR / "evaluation_pairs.csv"
)

print("Evaluation data loaded successfully.")

print("Shape:", evaluation_df.shape)

print("\nColumns:")
print(evaluation_df.columns.tolist())

print("\nDecision distribution:")
print(evaluation_df["decision"].value_counts())

Evaluation data loaded successfully.
Shape: (10174, 5)

Columns:
['pair_id', 'role', 'resume_text', 'job_description', 'decision']

Decision distribution:
decision
reject    5114
select    5060
Name: count, dtype: int64


In [9]:
# Cell 6 — Load matching model results

tfidf_test_df = pd.read_csv(
    RESULTS_DIR / "tfidf_test_results.csv"
)

tfidf_validation_df = pd.read_csv(
    RESULTS_DIR / "tfidf_validation_results.csv"
)

semantic_df = pd.read_csv(
    RESULTS_DIR / "semantic_results.csv"
)

hybrid_df = pd.read_csv(
    RESULTS_DIR / "hybrid_matching_results.csv"
)

print("All matching results loaded successfully.")

print("\nTF-IDF test:", tfidf_test_df.shape)
print("TF-IDF validation:", tfidf_validation_df.shape)
print("Semantic:", semantic_df.shape)
print("Hybrid:", hybrid_df.shape)

All matching results loaded successfully.

TF-IDF test: (2035, 8)
TF-IDF validation: (1628, 8)
Semantic: (10174, 4)
Hybrid: (10174, 21)


In [10]:
# Cell 7 — Inspect result columns

print("TF-IDF columns:")
print(tfidf_test_df.columns.tolist())

print("\nSemantic columns:")
print(semantic_df.columns.tolist())

print("\nHybrid columns:")
print(hybrid_df.columns.tolist())

TF-IDF columns:
['pair_id', 'role', 'resume_text', 'job_description', 'resume_group', 'jd_group', 'tfidf_similarity', 'tfidf_score']

Semantic columns:
['pair_id', 'role', 'semantic_similarity', 'semantic_score']

Hybrid columns:
['pair_id', 'role', 'resume_text', 'job_description', 'semantic_score', 'tfidf_score', 'resume_skills', 'jd_skills', 'matching_skills', 'missing_skills', 'skill_score', 'tfidf_normalized', 'semantic_normalized', 'skill_normalized', 'skill_available', 'hybrid_score', 'hybrid_score_100', 'tfidf_contribution', 'semantic_contribution', 'skill_contribution', 'match_category']


In [11]:
# Cell 8 — Prepare test evaluation labels

test_pair_ids = set(
    tfidf_test_df["pair_id"]
)

test_labels_df = evaluation_df[
    evaluation_df["pair_id"].isin(test_pair_ids)
].copy()

test_labels_df["y_true"] = (
    test_labels_df["decision"]
    .str.lower()
    .eq("select")
    .astype(int)
)

print("Test evaluation samples:", len(test_labels_df))

print("\nLabel distribution:")
print(test_labels_df["y_true"].value_counts())

Test evaluation samples: 2035

Label distribution:
y_true
0    1043
1     992
Name: count, dtype: int64


In [12]:
# Cell 9 — Align TF-IDF scores with evaluation labels

tfidf_eval = test_labels_df[
    ["pair_id", "y_true"]
].merge(
    tfidf_test_df[
        ["pair_id", "tfidf_score"]
    ],
    on="pair_id",
    how="inner"
)

print("TF-IDF evaluation rows:", len(tfidf_eval))

print("\nMissing TF-IDF scores:",
      tfidf_eval["tfidf_score"].isna().sum())

TF-IDF evaluation rows: 2035

Missing TF-IDF scores: 0


In [13]:
# Cell 10 — Align semantic and hybrid scores

semantic_eval = test_labels_df[
    ["pair_id", "y_true"]
].merge(
    semantic_df[
        ["pair_id", "semantic_score"]
    ],
    on="pair_id",
    how="inner"
)

hybrid_eval = test_labels_df[
    ["pair_id", "y_true"]
].merge(
    hybrid_df[
        ["pair_id", "hybrid_score_100"]
    ],
    on="pair_id",
    how="inner"
)

print("Semantic evaluation rows:", len(semantic_eval))
print("Hybrid evaluation rows:", len(hybrid_eval))

print("\nMissing semantic scores:",
      semantic_eval["semantic_score"].isna().sum())

print("Missing hybrid scores:",
      hybrid_eval["hybrid_score_100"].isna().sum())

Semantic evaluation rows: 2035
Hybrid evaluation rows: 2035

Missing semantic scores: 0
Missing hybrid scores: 0


In [14]:
# Cell 11 — Calculate test ROC-AUC

tfidf_auc = roc_auc_score(
    tfidf_eval["y_true"],
    tfidf_eval["tfidf_score"]
)

semantic_auc = roc_auc_score(
    semantic_eval["y_true"],
    semantic_eval["semantic_score"]
)

hybrid_auc = roc_auc_score(
    hybrid_eval["y_true"],
    hybrid_eval["hybrid_score_100"]
)

print("=" * 60)
print("TEST ROC-AUC")
print("=" * 60)

print(f"TF-IDF:   {tfidf_auc:.4f}")
print(f"Semantic: {semantic_auc:.4f}")
print(f"Hybrid:   {hybrid_auc:.4f}")

TEST ROC-AUC
TF-IDF:   0.4913
Semantic: 0.5124
Hybrid:   0.5034


In [15]:
# Cell 12 — Create model evaluation summary

evaluation_summary = pd.DataFrame({
    "Model": [
        "TF-IDF",
        "Semantic",
        "Hybrid"
    ],
    "Test_ROC_AUC": [
        tfidf_auc,
        semantic_auc,
        hybrid_auc
    ]
})

display(evaluation_summary)

,Model,Test_ROC_AUC
0,TF-IDF,0.491285
1,Semantic,0.512357
2,Hybrid,0.503362


In [16]:
# Cell 13 — Define transparent decision thresholds

TFIDF_THRESHOLD = tfidf_eval["tfidf_score"].median()
SEMANTIC_THRESHOLD = semantic_eval["semantic_score"].median()
HYBRID_THRESHOLD = hybrid_eval["hybrid_score_100"].median()

print("Thresholds based on the test-set medians:")
print(f"TF-IDF:   {TFIDF_THRESHOLD:.4f}")
print(f"Semantic: {SEMANTIC_THRESHOLD:.4f}")
print(f"Hybrid:   {HYBRID_THRESHOLD:.4f}")

Thresholds based on the test-set medians:
TF-IDF:   5.6824
Semantic: 53.1518
Hybrid:   46.1489


In [18]:
# Cell 14 — Calculate classification metrics

def classification_metrics(y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }


tfidf_metrics = classification_metrics(
    tfidf_eval["y_true"].values,
    tfidf_eval["tfidf_score"].values,
    TFIDF_THRESHOLD
)

semantic_metrics = classification_metrics(
    semantic_eval["y_true"].values,
    semantic_eval["semantic_score"].values,
    SEMANTIC_THRESHOLD
)

hybrid_metrics = classification_metrics(
    hybrid_eval["y_true"].values,
    hybrid_eval["hybrid_score_100"].values,
    HYBRID_THRESHOLD
)

classification_summary = pd.DataFrame(
    [
        {"Model": "TF-IDF", **tfidf_metrics},
        {"Model": "Semantic", **semantic_metrics},
        {"Model": "Hybrid", **hybrid_metrics}
    ]
)

display(classification_summary)

,Model,Accuracy,Precision,Recall,F1
0,TF-IDF,0.501720,0.489194,0.502016,0.495522
1,Semantic,0.521376,0.508841,0.522177,0.515423
2,Hybrid,0.519410,0.506876,0.520161,0.513433


In [19]:
# Cell 15 — Create confusion matrices

tfidf_pred = (
    tfidf_eval["tfidf_score"] >= TFIDF_THRESHOLD
).astype(int)

semantic_pred = (
    semantic_eval["semantic_score"] >= SEMANTIC_THRESHOLD
).astype(int)

hybrid_pred = (
    hybrid_eval["hybrid_score_100"] >= HYBRID_THRESHOLD
).astype(int)

tfidf_cm = confusion_matrix(
    tfidf_eval["y_true"],
    tfidf_pred
)

semantic_cm = confusion_matrix(
    semantic_eval["y_true"],
    semantic_pred
)

hybrid_cm = confusion_matrix(
    hybrid_eval["y_true"],
    hybrid_pred
)

print("TF-IDF confusion matrix:")
print(tfidf_cm)

print("\nSemantic confusion matrix:")
print(semantic_cm)

print("\nHybrid confusion matrix:")
print(hybrid_cm)

TF-IDF confusion matrix:
[[523 520]
 [494 498]]

Semantic confusion matrix:
[[543 500]
 [474 518]]

Hybrid confusion matrix:
[[541 502]
 [476 516]]


In [20]:
# Cell 16 — Plotting imports

import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve

print("Plotting libraries imported successfully.")

Plotting libraries imported successfully.


In [ ]:
# Cell 17 — Plot ROC curves

tfidf_fpr, tfidf_tpr, _ = roc_curve(
    tfidf_eval["y_true"],
    tfidf_eval["tfidf_score"]
)

semantic_fpr, semantic_tpr, _ = roc_curve(
    semantic_eval["y_true"],
    semantic_eval["semantic_score"]
)

hybrid_fpr, hybrid_tpr, _ = roc_curve(
    hybrid_eval["y_true"],
    hybrid_eval["hybrid_score_100"]
)

plt.figure(figsize=(8, 6))

plt.plot(
    tfidf_fpr,
    tfidf_tpr,
    label=f"TF-IDF (AUC = {tfidf_auc:.3f})"
)

plt.plot(
    semantic_fpr,
    semantic_tpr,
    label=f"Semantic (AUC = {semantic_auc:.3f})"
)

plt.plot(
    hybrid_fpr,
    hybrid_tpr,
    label=f"Hybrid (AUC = {hybrid_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random baseline"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Resume–Job Matching ROC Curves")
plt.legend()
plt.grid(True)
plt.show()

In [22]:
# Cell 18 — Load validated RAG output

rag_output_path = RESULTS_DIR / "validated_rag_output.json"

with open(rag_output_path, "r", encoding="utf-8") as f:
    rag_result = json.load(f)

print("Validated RAG output loaded successfully.")

print("\nFields:")
for key in rag_result:
    print("✓", key)

Validated RAG output loaded successfully.

Fields:
✓ matching_skills
✓ missing_skills
✓ relevant_experience
✓ education_alignment
✓ recommendations


In [23]:
# Cell 19 — Check RAG output structure

required_rag_fields = [
    "matching_skills",
    "missing_skills",
    "relevant_experience",
    "education_alignment",
    "recommendations"
]

missing_fields = [
    field
    for field in required_rag_fields
    if field not in rag_result
]

if not missing_fields:
    print("All required RAG fields are present.")
else:
    print("Missing fields:", missing_fields)

All required RAG fields are present.


In [24]:
# Cell 20 — Check RAG field types

expected_types = {
    "matching_skills": list,
    "missing_skills": list,
    "relevant_experience": list,
    "education_alignment": str,
    "recommendations": list
}

for field, expected_type in expected_types.items():
    actual_type = type(rag_result[field])

    status = "✓" if actual_type == expected_type else "✗"

    print(
        f"{status} {field}: "
        f"{actual_type.__name__}"
    )

✓ matching_skills: list
✓ missing_skills: list
✓ relevant_experience: list
✓ education_alignment: str
✓ recommendations: list


In [29]:
# Cell 21 — Compare deterministic and LLM skill extraction

def ensure_list(value):
    """Convert a list-like value into a Python list."""

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                return parsed

        except (ValueError, SyntaxError):
            pass

        return [value]

    return [value]


# Get the same demo pair used for the RAG analysis
demo_pair_id = hybrid_df["pair_id"].iloc[0]

# Get deterministic skill results from hybrid matching
deterministic_row = hybrid_df[
    hybrid_df["pair_id"] == demo_pair_id
].iloc[0]

deterministic_matching = set(
    ensure_list(deterministic_row["matching_skills"])
)

deterministic_missing = set(
    ensure_list(deterministic_row["missing_skills"])
)

# Get skills generated by the RAG/LLM system
llm_matching = set(
    ensure_list(rag_result["matching_skills"])
)

llm_missing = set(
    ensure_list(rag_result["missing_skills"])
)

print("Demo pair ID:", demo_pair_id)

print("\nDeterministic matching skills:")
print(deterministic_matching)

print("\nLLM matching skills:")
print(llm_matching)

print("\nDeterministic missing skills:")
print(deterministic_missing)

print("\nLLM missing skills:")
print(llm_missing)

Demo pair ID: P00001

Deterministic matching skills:
{'e-commerce'}

LLM matching skills:
{'e-commerce'}

Deterministic missing skills:
{'machine learning'}

LLM missing skills:
{'machine learning'}


In [30]:
# Cell 22 — Calculate skill consistency

matching_consistency = (
    len(deterministic_matching & llm_matching)
    / len(deterministic_matching)
    if deterministic_matching
    else 1.0
)

missing_consistency = (
    len(deterministic_missing & llm_missing)
    / len(deterministic_missing)
    if deterministic_missing
    else 1.0
)

print(f"Matching-skill consistency: {matching_consistency:.2%}")
print(f"Missing-skill consistency: {missing_consistency:.2%}")

Matching-skill consistency: 100.00%
Missing-skill consistency: 100.00%


In [ ]:
# Cell 23 — Reconstruct and inspect retrieved context

print("=" * 70)
print("RAG RETRIEVAL EVALUATION")
print("=" * 70)

# Load the demo resume and job description
demo_pair_id = hybrid_df["pair_id"].iloc[0]

demo_row = evaluation_df[
    evaluation_df["pair_id"] == demo_pair_id
]

if demo_row.empty:
    print("Demo pair not found in evaluation data.")
else:
    demo_row = demo_row.iloc[0]

    resume_text = str(demo_row["resume_text"])
    job_description = str(demo_row["job_description"])

    print("\nDemo pair ID:", demo_pair_id)
    print("Resume length:", len(resume_text))
    print("Job description length:", len(job_description))

    print("\nResume preview:")
    print(resume_text[:500])

    print("\nJob description preview:")
    print(job_description[:500])

In [34]:
# Cell 24 — Check retrieval coverage

print("=" * 70)
print("RETRIEVAL COVERAGE CHECK")
print("=" * 70)

# Use the resume text reconstructed in Cell 23
resume_text_for_check = resume_text.strip()

# For this evaluation, use the retrieved context if it exists.
# If it does not exist, use the resume text as the available
# reference context.
if "retrieved_context" in globals():
    context_for_check = retrieved_context
else:
    context_for_check = resume_text_for_check

retrieval_coverage = (
    len(context_for_check) / len(resume_text_for_check)
    if len(resume_text_for_check) > 0
    else 0
)

print(f"Original resume length: {len(resume_text_for_check)} characters")
print(f"Context length: {len(context_for_check)} characters")
print(f"Context / resume length: {retrieval_coverage:.2%}")

print("\nNote:")
print(
    "This is a simple coverage diagnostic. "
    "It is not a formal retrieval-quality metric."
)

RETRIEVAL COVERAGE CHECK
Original resume length: 2595 characters
Context length: 2595 characters
Context / resume length: 100.00%

Note:
This is a simple coverage diagnostic. It is not a formal retrieval-quality metric.


In [36]:
# Cell 25 — Load validated RAG output

rag_output_path = RESULTS_DIR / "validated_rag_output.json"

with open(rag_output_path, "r", encoding="utf-8") as f:
    rag_result = json.load(f)

print("Validated RAG output loaded successfully.")

print("\nAvailable fields:")
for key in rag_result:
    print("✓", key)

Validated RAG output loaded successfully.

Available fields:
✓ matching_skills
✓ missing_skills
✓ relevant_experience
✓ education_alignment
✓ recommendations


In [37]:
# Cell 26 — Validate RAG output structure

required_rag_fields = [
    "matching_skills",
    "missing_skills",
    "relevant_experience",
    "education_alignment",
    "recommendations"
]

missing_fields = [
    field
    for field in required_rag_fields
    if field not in rag_result
]

if not missing_fields:
    print("All required RAG fields are present.")
else:
    print("Missing fields:")
    print(missing_fields)

All required RAG fields are present.


In [38]:
# Cell 27 — Check RAG field types

expected_types = {
    "matching_skills": list,
    "missing_skills": list,
    "relevant_experience": list,
    "education_alignment": str,
    "recommendations": list
}

for field, expected_type in expected_types.items():

    actual_type = type(rag_result[field])

    status = "✓" if actual_type == expected_type else "✗"

    print(
        f"{status} {field}: "
        f"{actual_type.__name__}"
    )

✓ matching_skills: list
✓ missing_skills: list
✓ relevant_experience: list
✓ education_alignment: str
✓ recommendations: list


In [39]:
# Cell 28 — Compare deterministic and LLM skill extraction

def ensure_list(value):
    """Convert a list-like value into a Python list."""

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                return parsed

        except (ValueError, SyntaxError):
            pass

        return [value]

    return [value]


# Use the same demo pair used for the RAG example
demo_pair_id = hybrid_df["pair_id"].iloc[0]

# Get deterministic skill results
deterministic_row = hybrid_df[
    hybrid_df["pair_id"] == demo_pair_id
].iloc[0]

deterministic_matching = set(
    ensure_list(deterministic_row["matching_skills"])
)

deterministic_missing = set(
    ensure_list(deterministic_row["missing_skills"])
)

# Get LLM skill results
llm_matching = set(
    ensure_list(rag_result["matching_skills"])
)

llm_missing = set(
    ensure_list(rag_result["missing_skills"])
)

print("Demo pair ID:", demo_pair_id)

print("\nDeterministic matching skills:")
print(deterministic_matching)

print("\nLLM matching skills:")
print(llm_matching)

print("\nDeterministic missing skills:")
print(deterministic_missing)

print("\nLLM missing skills:")
print(llm_missing)

Demo pair ID: P00001

Deterministic matching skills:
{'e-commerce'}

LLM matching skills:
{'e-commerce'}

Deterministic missing skills:
{'machine learning'}

LLM missing skills:
{'machine learning'}


In [40]:
# Cell 29 — Calculate skill consistency

matching_consistency = (
    len(deterministic_matching & llm_matching)
    / len(deterministic_matching)
    if deterministic_matching
    else 1.0
)

missing_consistency = (
    len(deterministic_missing & llm_missing)
    / len(deterministic_missing)
    if deterministic_missing
    else 1.0
)

print(
    f"Matching-skill consistency: "
    f"{matching_consistency:.2%}"
)

print(
    f"Missing-skill consistency: "
    f"{missing_consistency:.2%}"
)

Matching-skill consistency: 100.00%
Missing-skill consistency: 100.00%


In [41]:
# Cell 30 — Load demo resume and job description

demo_row = evaluation_df[
    evaluation_df["pair_id"] == demo_pair_id
]

if demo_row.empty:
    raise ValueError(
        f"Could not find pair_id {demo_pair_id} "
        "in evaluation_pairs.csv"
    )

demo_row = demo_row.iloc[0]

resume_text = str(
    demo_row["resume_text"]
).strip()

job_description = str(
    demo_row["job_description"]
).strip()

print("Demo pair loaded successfully.")

print("\nPair ID:", demo_pair_id)

print("\nResume length:",
      len(resume_text), "characters")

print("Job description length:",
      len(job_description), "characters")

Demo pair loaded successfully.

Pair ID: P00001

Resume length: 2595 characters
Job description length: 137 characters


In [42]:
# Cell 31 — Inspect source documents

print("=" * 70)
print("RESUME PREVIEW")
print("=" * 70)

print(resume_text[:1000])

print("\n" + "=" * 70)
print("JOB DESCRIPTION PREVIEW")
print("=" * 70)

print(job_description[:1000])

RESUME PREVIEW
Here's a professional resume for Jason Jones: Jason Jones E-commerce Specialist Contact Information: * Email: [jasonjones@email.com](mailto:jasonjones@email.com) * Phone: 555-123-4567 * LinkedIn: linkedin.com/in/jasonjones Summary: Results-driven E-commerce Specialist with 5+ years of experience in inventory management, SEO, online advertising, and analytics. Proven track record of increasing online sales, improving website traffic, and optimizing inventory levels. Skilled in analyzing complex data sets, identifying trends, and making data-driven decisions. Passionate about staying up-to-date with the latest e-commerce trends and technologies. Professional Experience: E-commerce Specialist, XYZ Corporation (2018-Present) * Managed inventory levels across multiple channels, resulting in a 25% reduction in stockouts and a 15% reduction in overstocking * Developed and implemented SEO strategies that increased website traffic by 30% and improved search engine rankings by 20%

In [43]:
# Cell 32 — Retrieval evaluation setup

from sentence_transformers import SentenceTransformer
import faiss
import re

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [44]:
# Cell 33 — Sentence-aware chunking

def chunk_text(text, chunk_size=500, overlap=100):

    text = str(text).strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [text]

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    chunks = []
    current_chunk = ""

    for sentence in sentences:

        sentence = sentence.strip()

        if not sentence:
            continue

        if (
            len(current_chunk)
            + len(sentence)
            + 1
            <= chunk_size
        ):
            current_chunk += (
                " " if current_chunk else ""
            ) + sentence

        else:

            if current_chunk:
                chunks.append(
                    current_chunk.strip()
                )

            current_chunk = sentence

    if current_chunk:
        chunks.append(
            current_chunk.strip()
        )

    return chunks


resume_chunks_eval = chunk_text(
    resume_text
)

print("Resume chunks created:",
      len(resume_chunks_eval))

for i, chunk in enumerate(
    resume_chunks_eval,
    start=1
):
    print(
        f"Chunk {i}: "
        f"{len(chunk)} characters"
    )

Resume chunks created: 3
Chunk 1: 469 characters
Chunk 2: 181 characters
Chunk 3: 1943 characters


In [45]:
# Cell 34 — Create resume embeddings and FAISS index

resume_embeddings_eval = embedding_model.encode(
    resume_chunks_eval,
    normalize_embeddings=True,
    convert_to_numpy=True
)

embedding_dimension = (
    resume_embeddings_eval.shape[1]
)

resume_index_eval = faiss.IndexFlatIP(
    embedding_dimension
)

resume_index_eval.add(
    resume_embeddings_eval.astype("float32")
)

print("FAISS retrieval system created.")

print(
    "Embedding dimension:",
    embedding_dimension
)

print(
    "Indexed chunks:",
    resume_index_eval.ntotal
)

FAISS retrieval system created.
Embedding dimension: 384
Indexed chunks: 3


In [46]:
# Cell 35 — Retrieve relevant resume chunks

jd_embedding_eval = embedding_model.encode(
    [job_description],
    normalize_embeddings=True,
    convert_to_numpy=True
)

top_k = min(
    3,
    len(resume_chunks_eval)
)

similarities_eval, indices_eval = (
    resume_index_eval.search(
        jd_embedding_eval.astype("float32"),
        top_k
    )
)

retrieved_chunks_eval = [
    resume_chunks_eval[index]
    for index in indices_eval[0]
]

retrieved_context_eval = "\n\n".join(
    retrieved_chunks_eval
)

print("Retrieved chunks:",
      len(retrieved_chunks_eval))

for rank, (score, index) in enumerate(
    zip(
        similarities_eval[0],
        indices_eval[0]
    ),
    start=1
):

    print("\n" + "-" * 70)
    print("Rank:", rank)
    print("Similarity:", f"{score:.4f}")
    print("Chunk index:", index)
    print("Text:")
    print(resume_chunks_eval[index])

Retrieved chunks: 3

----------------------------------------------------------------------
Rank: 1
Similarity: 0.6150
Chunk index: 1
Text:
Skilled in analyzing complex data sets, identifying trends, and making data-driven decisions. Passionate about staying up-to-date with the latest e-commerce trends and technologies.

----------------------------------------------------------------------
Rank: 2
Similarity: 0.4913
Chunk index: 2
Text:
Professional Experience: E-commerce Specialist, XYZ Corporation (2018-Present) * Managed inventory levels across multiple channels, resulting in a 25% reduction in stockouts and a 15% reduction in overstocking * Developed and implemented SEO strategies that increased website traffic by 30% and improved search engine rankings by 20% * Created and executed online advertising campaigns that generated a 50% increase in sales and a 20% increase in conversion rates * Analyzed website analytics to identify trends, optimize user experience, and improve custome

In [47]:
# Cell 36 — Evaluate retrieval coverage

retrieved_context_length = len(
    retrieved_context_eval
)

resume_length = len(
    resume_text
)

retrieval_coverage = (
    retrieved_context_length / resume_length
    if resume_length > 0
    else 0
)

print("=" * 70)
print("RETRIEVAL COVERAGE")
print("=" * 70)

print(
    "Original resume length:",
    resume_length,
    "characters"
)

print(
    "Retrieved context length:",
    retrieved_context_length,
    "characters"
)

print(
    "Context / resume length:",
    f"{retrieval_coverage:.2%}"
)

print(
    "\nNote: This is a coverage diagnostic, "
    "not a formal retrieval-quality score."
)

RETRIEVAL COVERAGE
Original resume length: 2595 characters
Retrieved context length: 2597 characters
Context / resume length: 100.08%

Note: This is a coverage diagnostic, not a formal retrieval-quality score.


In [48]:
# Cell 37 — Check experience grounding

retrieved_context_lower = (
    retrieved_context_eval.lower()
)

experience_grounding = []

for experience in rag_result[
    "relevant_experience"
]:

    experience_lower = (
        experience.lower()
    )

    words = [
        word.strip(".,()")
        for word in experience_lower.split()
        if len(word.strip(".,()")) > 3
    ]

    matched_words = [
        word
        for word in words
        if word in retrieved_context_lower
    ]

    ratio = (
        len(matched_words) / len(words)
        if words
        else 0
    )

    experience_grounding.append({
        "statement": experience,
        "matched_word_ratio": ratio
    })

display(
    pd.DataFrame(
        experience_grounding
    )
)

,statement,matched_word_ratio
0,"E-commerce Specialist, XYZ Corporation (2018-P...",1.0
1,"E-commerce Coordinator, ABC Retail (2015-2018)",1.0


In [49]:
# Cell 38 — Create final evaluation report

final_evaluation = {
    "traditional_matching": {
        "tfidf_test_roc_auc": float(
            tfidf_auc
        ),
        "semantic_test_roc_auc": float(
            semantic_auc
        ),
        "hybrid_test_roc_auc": float(
            hybrid_auc
        )
    },

    "classification_metrics":
        classification_summary.to_dict(
            orient="records"
        ),

    "rag_structure_valid":
        len(missing_fields) == 0,

    "matching_skill_consistency":
        float(matching_consistency),

    "missing_skill_consistency":
        float(missing_consistency),

    "retrieved_chunks":
        len(retrieved_chunks_eval),

    "retrieval_coverage":
        float(retrieval_coverage)
}

print(
    "Final evaluation report created successfully."
)

print(
    json.dumps(
        final_evaluation,
        indent=4
    )
)

Final evaluation report created successfully.
{
    "traditional_matching": {
        "tfidf_test_roc_auc": 0.4912845428818854,
        "semantic_test_roc_auc": 0.5123567639872576,
        "hybrid_test_roc_auc": 0.5033624702316519
    },
    "classification_metrics": [
        {
            "Model": "TF-IDF",
            "Accuracy": 0.5017199017199018,
            "Precision": 0.48919449901768175,
            "Recall": 0.5020161290322581,
            "F1": 0.4955223880597015
        },
        {
            "Model": "Semantic",
            "Accuracy": 0.5213759213759214,
            "Precision": 0.5088408644400786,
            "Recall": 0.5221774193548387,
            "F1": 0.5154228855721393
        },
        {
            "Model": "Hybrid",
            "Accuracy": 0.5194103194103195,
            "Precision": 0.5068762278978389,
            "Recall": 0.5201612903225806,
            "F1": 0.5134328358208955
        }
    ],
    "rag_structure_valid": true,
    "matching_skill_consiste

In [50]:
# Cell 39 — Save final evaluation report

evaluation_output_path = (
    RESULTS_DIR /
    "final_evaluation_report.json"
)

with open(
    evaluation_output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_evaluation,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Evaluation report saved successfully."
)

print(
    "Saved to:",
    evaluation_output_path.resolve()
)

Evaluation report saved successfully.
Saved to: C:\Users\Me\OneDrive\Desktop\Resume project ( Gen AI-RAG)\results\final_evaluation_report.json


In [51]:
# Cell 40A — Verify final evaluation report

report_path = RESULTS_DIR / "final_evaluation_report.json"

with open(report_path, "r", encoding="utf-8") as f:
    evaluation_report = json.load(f)

print("=" * 70)
print("FINAL EVALUATION REPORT")
print("=" * 70)

print("\nTraditional matching:")

for model, score in evaluation_report["traditional_matching"].items():
    print(f"{model}: {score:.4f}")

print("\nRAG structure valid:",
      evaluation_report["rag_structure_valid"])

print(
    "Matching-skill consistency:",
    f"{evaluation_report['matching_skill_consistency']:.2%}"
)

print(
    "Missing-skill consistency:",
    f"{evaluation_report['missing_skill_consistency']:.2%}"
)

print(
    "Retrieved chunks:",
    evaluation_report["retrieved_chunks"]
)

print(
    "Retrieval coverage:",
    f"{evaluation_report['retrieval_coverage']:.2%}"
)

FINAL EVALUATION REPORT

Traditional matching:
tfidf_test_roc_auc: 0.4913
semantic_test_roc_auc: 0.5124
hybrid_test_roc_auc: 0.5034

RAG structure valid: True
Matching-skill consistency: 100.00%
Missing-skill consistency: 100.00%
Retrieved chunks: 3
Retrieval coverage: 100.08%
